Vid Analysis

In [72]:
import tensordict
import torch

td = tensordict.load("/home/jacopo/PycharmProjects/progetto-tesi/data/EEGAVI/AMIGOS/interleaved/200/")
fp32 = td["vid"]["data"]
print(fp32.shape)
b, t, p, d = fp32.shape
fp32 = fp32.reshape(b * t * p, d)

norms = fp32.norm(dim=1)
mask = norms > 0

fp32 = fp32[mask]

print(fp32.dtype)
fp32 = torch.nn.functional.normalize(fp32.float(), dim=1)
sim = (fp32 * fp32).sum(dim=1).mean()
print(sim)


def quantize_int8_per_dim(x: torch.Tensor, eps=1e-8):
    x = x.float()
    scale = x.abs().amax(dim=0) / 127.0  # [D]
    scale = torch.clamp(scale, min=eps)
    q = torch.clamp((x / scale).round(), -127, 127).to(torch.int8)
    return q, scale.half()


def dequantize_int8_per_dim(q: torch.Tensor, scale: torch.Tensor):
    return q.float() * scale.float()


fp8 = td["vid"]["data"]
fp8 = fp8.reshape(b * t * p, d)

fp8 = fp8[mask]
fp8, s = quantize_int8_per_dim(fp8)
fp8 = dequantize_int8_per_dim(fp8, s)
fp8 = torch.nn.functional.normalize(fp8.float(), dim=1)

cos = (fp32 * fp8).sum(dim=1)
cos
print("median:", cos.median().item())
print("p1:", cos.quantile(0.01).item())
print("min:", cos.min().item())
print("std:", cos.std().item())
print("similarity", cos.mean())

torch.Size([164, 8, 199, 768])
torch.float32
tensor(1.)


RuntimeError: shape '[261088, 768]' is invalid for input of size 403046400

In [157]:
# todo quantiaziont lib
def quantize_int8_per_dim(x: torch.Tensor, eps=1e-8):
    x = x.float()
    scale = x.abs().amax(dim=0) / 127.0  # [D]
    scale = torch.clamp(scale, min=eps)
    q = torch.clamp((x / scale).round(), -127, 127).to(torch.int8)
    return q, scale.half()


def quantize_int8_per_row(x, eps=1e-8):
    x = x.float()
    s = x.abs().amax(dim=-1, keepdim=True) / 127.0
    s = s.clamp_min(eps)
    q = (x / s).round().clamp(-127, 127).to(torch.int8)
    return q, s.half()


def dequantize_int8_per_row(q, s):
    return q.float() * s.float()


def dequantize_int8_per_dim(q: torch.Tensor, scale: torch.Tensor):
    return q.float() * scale.float()

In [158]:
import tensordict
import torch

# Text experiment tensor
tensor_path = "/home/jacopo/PycharmProjects/progetto-tesi/data/EEGAVI/AMIGOS/interleaved/200/"
td = tensordict.load(tensor_path)

for modality in ['eeg', 'vid', 'aud', 'ecg', 'txt']:
    fp32 = td[modality]["data"]
    print("---------", modality, "---------")
    print("Modality shape:", fp32.shape)
    fp32 = fp32.reshape(-1, fp32.shape[-1])
    norms = fp32.norm(dim=1)
    mask = norms > 0
    # Only valid rows
    fp32 = fp32[mask]
    fp32 = torch.nn.functional.normalize(fp32, dim=1)
    sim = (fp32 * fp32).sum(dim=1).mean()
    print("Sanity check self similarity:", sim)
    # Take the float32 -> int + float16
    fp8, s = quantize_int8_per_row(td[modality]["data"])
    fp8 = dequantize_int8_per_dim(fp8, s)

    fp8 = fp8.reshape(-1, fp32.shape[-1])
    fp8 = fp8[mask]
    fp8 = torch.nn.functional.normalize(fp8.float(), dim=1)

    cos = (fp32 * fp8).sum(dim=1)
    print("median:", cos.median().item())
    print("p1:", cos.quantile(0.01).item())
    print("min:", cos.min().item())
    print("std:", cos.std().item())
    print("similarity", cos.mean())
    print()

--------- eeg ---------
Modality shape: torch.Size([164, 32, 34, 200])
Sanity check self similarity: tensor(1.)
median: 0.9999790787696838
p1: 0.9999597072601318
min: 0.9999254941940308
std: 5.763442459283397e-06
similarity tensor(1.0000)

--------- vid ---------
Modality shape: torch.Size([164, 8, 400, 768])
Sanity check self similarity: tensor(1.)
median: 0.9998829364776611
p1: 0.9991883039474487
min: 0.9984341263771057
std: 0.0001322182361036539
similarity tensor(0.9999)

--------- aud ---------
Modality shape: torch.Size([164, 8, 199, 768])
Sanity check self similarity: tensor(1.)
median: 0.9998286366462708
p1: 0.9995434284210205
min: 0.9992691874504089
std: 8.069592149695382e-05
similarity tensor(0.9998)

--------- ecg ---------
Modality shape: torch.Size([164, 8, 32, 256])
Sanity check self similarity: tensor(1.)
median: 0.9999669790267944
p1: 0.9999186992645264
min: 0.9997878670692444
std: 1.3736046639678534e-05
similarity tensor(1.0000)

--------- txt ---------
Modality shape: 

In [61]:
cos.numel() == 1

True

In [69]:
td['vid']['data'] = td['vid']['data'].half()

In [71]:
tensordict.save(td, "/tmp/285", copy_existing=True)

TensorDict(
    fields={
        assessment: MemoryMappedTensor(shape=torch.Size([164, 4]), device=cpu, dtype=torch.float64, is_shared=True),
        aud: TensorDict(
            fields={
                data: MemoryMappedTensor(shape=torch.Size([164, 8, 199, 768]), device=cpu, dtype=torch.float32, is_shared=True),
                mask: MemoryMappedTensor(shape=torch.Size([164, 8]), device=cpu, dtype=torch.bool, is_shared=True)},
            batch_size=torch.Size([164]),
            device=cpu,
            is_shared=False),
        ecg: TensorDict(
            fields={
                data: MemoryMappedTensor(shape=torch.Size([164, 8, 32, 256]), device=cpu, dtype=torch.float32, is_shared=True),
                mask: MemoryMappedTensor(shape=torch.Size([164, 8]), device=cpu, dtype=torch.bool, is_shared=True)},
            batch_size=torch.Size([164]),
            device=cpu,
            is_shared=False),
        eeg: TensorDict(
            fields={
                data: MemoryMappedTen

In [82]:
td

TensorDict(
    fields={
        assessment: MemoryMappedTensor(shape=torch.Size([164, 4]), device=cpu, dtype=torch.float64, is_shared=True),
        aud: TensorDict(
            fields={
                data: MemoryMappedTensor(shape=torch.Size([164, 8, 199, 768]), device=cpu, dtype=torch.float32, is_shared=True),
                mask: MemoryMappedTensor(shape=torch.Size([164, 8]), device=cpu, dtype=torch.bool, is_shared=True)},
            batch_size=torch.Size([164]),
            device=cpu,
            is_shared=False),
        ecg: TensorDict(
            fields={
                data: MemoryMappedTensor(shape=torch.Size([164, 8, 32, 256]), device=cpu, dtype=torch.float32, is_shared=True),
                mask: MemoryMappedTensor(shape=torch.Size([164, 8]), device=cpu, dtype=torch.bool, is_shared=True)},
            batch_size=torch.Size([164]),
            device=cpu,
            is_shared=False),
        eeg: TensorDict(
            fields={
                data: MemoryMappedTen

In [103]:
quantize_int8_per_dim(fp8)[0][0]

tensor([-127,  127, -127, -127,  127,  127,  127,  127, -127, -127,  127, -127,
         127, -127,  127, -127, -127, -127, -127,  127, -127,  127,  127, -127,
        -127, -127, -127,  127,  127, -127, -127,  127,  127, -127, -127,  127,
        -127, -127,  127,  127,  127, -127, -127,  127,  127, -127,  127,  127,
         127, -127, -127, -127, -127, -127,  127, -127,  127, -127,  127, -127,
         127, -127, -127,  127, -127,  127,  127, -127, -127,  127,  127, -127,
         127, -127,  127,  127,  127,  127,  127,  127,  127,  127, -127, -127,
        -127, -127, -127, -127, -127,  127, -127, -127,  127, -127, -127, -127,
         127,  127, -127,  127,  127,  127,  127, -127,  127, -127, -127, -127,
         127,  127, -127, -127,  127,  127,  127,  127, -127, -127,  127,  127,
        -127, -127,  127,  127,  127, -127,  127,    0,  127,  127,  127, -127,
         127,  127,  127, -127, -127,  127,  127,  127, -127,  127,  127,  127,
         127,  127, -127,  127, -127, -1

What now:
- Remvoe assessments
- Turn all into int8 + scale (x4 reduction of space)

In [104]:
fp8[0]

tensor([-1.1881e-01,  4.8295e-02, -2.5510e-03, -1.1014e-02,  5.1959e-02,
         1.0295e-02,  1.1542e-01,  7.0399e-04, -8.5932e-02, -7.0671e-02,
         1.3323e-03, -3.5487e-02,  1.8440e-02, -6.7371e-03,  2.4405e-02,
        -2.9507e-02, -5.8136e-02, -5.0445e-02, -2.0771e-02,  2.9038e-02,
        -6.3646e-02,  2.4026e-02,  2.6237e-02, -6.0406e-03, -1.1075e-02,
        -1.4004e-03, -1.8622e-02,  3.2762e-02,  2.8841e-03, -5.6955e-02,
        -4.3935e-02,  2.5419e-02,  8.7930e-02, -2.4995e-02, -3.6698e-02,
         6.2450e-03, -6.6493e-02, -6.7159e-02,  2.0559e-02,  4.2391e-02,
         2.1877e-02, -4.2875e-02, -3.4367e-02,  6.1466e-02,  6.5645e-02,
        -7.8544e-02,  2.9492e-02,  1.0802e-02,  6.3344e-02, -4.5085e-02,
        -1.8228e-02, -2.7720e-02, -3.6713e-03, -3.6607e-02,  5.4260e-02,
        -2.0862e-02,  1.5033e-02, -6.0104e-02,  1.6396e-02, -3.3246e-02,
         1.7501e-02, -5.9801e-04, -1.6351e-01,  8.4902e-02, -7.5818e-02,
         1.6108e-02,  4.8386e-02, -7.6000e-03, -2.4

In [105]:
a, b = quantize_int8_per_dim(fp8)

In [107]:
(a * b)[0]

tensor([-1.1884e-01,  4.8309e-02, -2.5501e-03, -1.1017e-02,  5.1971e-02,
         1.0292e-02,  1.1542e-01,  7.0381e-04, -8.5938e-02, -7.0679e-02,
         1.3323e-03, -3.5492e-02,  1.8433e-02, -6.7368e-03,  2.4399e-02,
        -2.9510e-02, -5.8136e-02, -5.0446e-02, -2.0767e-02,  2.9037e-02,
        -6.3660e-02,  2.4033e-02,  2.6230e-02, -6.0425e-03, -1.1078e-02,
        -1.4000e-03, -1.8616e-02,  3.2776e-02,  2.8839e-03, -5.6946e-02,
        -4.3945e-02,  2.5421e-02,  8.7952e-02, -2.4994e-02, -3.6713e-02,
         6.2447e-03, -6.6467e-02, -6.7139e-02,  2.0554e-02,  4.2389e-02,
         2.1881e-02, -4.2877e-02, -3.4363e-02,  6.1462e-02,  6.5674e-02,
        -7.8552e-02,  2.9495e-02,  1.0803e-02,  6.3354e-02, -4.5074e-02,
        -1.8234e-02, -2.7725e-02, -3.6716e-03, -3.6621e-02,  5.4260e-02,
        -2.0859e-02,  1.5030e-02, -6.0089e-02,  1.6403e-02, -3.3234e-02,
         1.7502e-02, -5.9795e-04, -1.6345e-01,  8.4900e-02, -7.5806e-02,
         1.6113e-02,  4.8401e-02, -7.5989e-03, -2.4

In [109]:
sat = (a.abs() == 127).float().mean()

In [111]:
print("sat fraction:", (a.abs() == 127).float().mean().item())
print("worst-dim sat:", (a.abs() == 127).float().mean(dim=0).max().item())

sat fraction: 0.9921875
worst-dim sat: 1.0


In [143]:
fp32 = td["vid"]["data"]
b4, s = quantize_int8_per_row(fp32)
after = dequantize_int8_per_dim(b4, s)

In [144]:
fp32

MemoryMappedTensor([[[[ 4.3801e-01,  1.2455e-02, -5.2842e-01,  ...,
                        7.2826e-01,  7.8793e-01,  4.8396e-01],
                      [ 5.6954e-01, -8.6913e-02, -4.3623e-01,  ...,
                        6.1187e-01,  8.1778e-01,  5.8626e-01],
                      [ 8.6797e-01,  2.6727e-01,  5.9613e-02,  ...,
                        4.3829e-01,  7.6115e-01,  4.8219e-01],
                      ...,
                      [ 3.5509e-01, -9.0559e-02, -4.4315e-01,  ...,
                        3.8108e-01,  3.2698e-01,  1.8092e-01],
                      [ 4.7734e-01,  5.8525e-02, -4.3121e-01,  ...,
                        3.6775e-01,  4.3499e-01,  4.6116e-01],
                      [ 7.0445e-01,  1.2997e-01, -4.9360e-01,  ...,
                        5.8213e-01,  5.3586e-01,  4.5780e-01]],

                     [[ 3.6329e-01,  6.1996e-02, -5.2428e-01,  ...,
                        5.9464e-01,  7.5177e-01,  4.4684e-01],
                      [ 4.7544e-01, -2.0095e-02, -3.44

In [145]:
b4

tensor([[[[ 127,   26, -127,  ...,  127,  127,  127],
          [ 127, -127, -127,  ...,  127,  127,  127],
          [ 127,   92,   38,  ...,  127,  127,  127],
          ...,
          [  88, -127,  -99,  ...,  114,  107,   61],
          [ 127,   29, -127,  ...,  127,  127,  127],
          [ 127,   91, -127,  ...,  127,  127,  127]],

         [[ 105,  127, -126,  ...,  104,  121,  117],
          [ 106,  -29, -100,  ...,   96,  107,  109],
          [ 115,  127,  127,  ...,   79,  118,   87],
          ...,
          [ 127,    4, -127,  ...,  127,  127,  127],
          [ 120,  127, -102,  ...,   51,   94,   71],
          [ 125,  127,  -95,  ...,   97,   95,  124]],

         [[   0,    0,    0,  ...,    0,    0,    0],
          [   0,    0,    0,  ...,    0,    0,    0],
          [   0,    0,    0,  ...,    0,    0,    0],
          ...,
          [   0,    0,    0,  ...,    0,    0,    0],
          [   0,    0,    0,  ...,    0,    0,    0],
          [   0,    0,    0,  ...

In [151]:
after[0][1]

tensor([[ 0.3621,  0.0620, -0.5244,  ...,  0.5963,  0.7505,  0.4459],
        [ 0.4755, -0.0198, -0.3435,  ...,  0.4625,  0.6890,  0.5031],
        [ 0.7861,  0.3706,  0.1994,  ...,  0.2726,  0.7072,  0.3304],
        ...,
        [ 0.5101,  0.0029, -0.5692,  ...,  0.4244,  0.3871,  0.3750],
        [ 0.4511,  0.2565, -0.3463,  ...,  0.1477,  0.3220,  0.2578],
        [ 0.6933,  0.1818, -0.3693,  ...,  0.4448,  0.4008,  0.4470]])

In [153]:
fp32[0][1][0]

MemoryMappedTensor([ 3.6329e-01,  6.1996e-02, -5.2428e-01, -1.2874e+00,
                     1.5869e+00, -4.6591e-02, -1.6058e-01,  9.6055e-01,
                    -3.1213e-01,  5.6936e-01,  2.4518e-01,  6.2031e-01,
                     3.1257e-01,  5.6572e-01,  2.5826e-01,  4.2984e-01,
                     1.4161e+00,  4.6033e-01,  8.3215e-01,  2.0775e-01,
                    -5.0746e-01, -5.4059e-01, -1.3191e-01,  4.3817e-01,
                    -1.2882e-01, -4.5868e-01, -2.4148e-01, -5.9664e-01,
                     2.8783e-01,  1.5477e+00, -2.9517e-01,  6.9030e-01,
                    -9.0433e-01,  2.1147e-01, -7.7059e-01, -2.8141e-01,
                     4.0208e-01,  1.8283e+00, -9.7960e-01,  3.5308e-01,
                    -9.2768e-01,  7.0829e-01,  4.8724e-01, -6.5473e-01,
                    -4.5837e-01,  1.0663e+00,  2.3633e-01, -1.1093e+00,
                     1.1033e+00, -1.0135e-01,  1.1240e+00,  7.2231e-01,
                    -1.1651e+00, -5.0122e-01,  5.3408e-01, -4.52

In [154]:
import torch.nn.functional as F

cos_sim = (F.normalize(fp32[0][1], dim=-1) * F.normalize(after[0][1], dim=-1)).sum(dim=-1)
cos_dist = 1 - cos_sim

In [156]:
cos_dist.mean()

tensor(1.4590e-06)